# **Setup Documentation**

> #### RaspberryPi Configurations

**Step 1** ::: Check if the system is up-to-date and upgrade.

In [ ]:
sudo apt update
sudo apt upgrade -y

**Step 2** ::: Setup the usernames and hostnames. It is strongly recommended that the username is ``idmind``. 

In [ ]:
sudo hostnamectl set-hostname tagi && sudo sed -i "s/127.0.1.1.*/127.0.1.1\ttagi/g" /etc/hosts
sudo loginctl enable-linger idmind

###### <u>Note</u>: This should have been already done on Raspberry Pi OS Config.

**Step 3** ::: Configure I2C (Inter-Integrated Circuit) Protocol and make boot happen without the user interface.

In [ ]:
sudo raspi-config nonint do_i2c 0  
sudo raspi-config nonint do_boot_behaviour B2

**Step 4** ::: Edit ``config.txt``, on ``/boot/firmware/``. At the very bottom, add the following lines to take the biggest part of the Raspberry Pi capabilities:

In [ ]:
enable_uart=1
dtparam=uart0=on
dtoverlay=disable-bt
disable_splash=1
dtoverlay=gpio-shutdown,gpio_pin=27,active_low=0
dtoverlay=gpio-poweroff,gpiopin=4,active_low=1

> #### Dependencies installation

**Step 1** ::: Run this command to install dependencies for some Python packages.

In [ ]:
sudo apt install -y libzbar0 portaudio19-dev

**Step 2** ::: Run this command inside elmo-v2's folder to syncronize most of the libraries needed.

In [ ]:
uv venv --python 3.13 --system-site-packages
uv pip install -r requirements.txt

###### <u>Note</u>: Before this command, download uv. Check [Tools](tools.ipynb) for useful information. 

**Step 3** ::: Run this command to install Neopixel, absolutely necessary for the 13x13 LED Matrix to work, and other dependencies.

In [ ]:
sudo apt update
sudo apt upgrade
sudo apt install python3-pip -y
sudo pip install rpi_ws281x adafruit-circuitpython-neopixel --break-system-packages -y
sudo apt install mpg123 -y
sudo apt install libcap-dev swig liblgpio-dev
sudo apt install libcamera-dev
sudo apt install sox -y
sudo apt install wtype -y

> #### Kiosk Settings

##### __**Splash Screen**__

**Step 1** ::: Copy your image to the Plymouth theme.

In [ ]:
sudo mkdir -p /usr/share/plymouth/themes/pix
sudo cp CUSTOM_IMAGE.png /usr/share/plymouth/themes/pix/splash.png

**Step 2** ::: Open ``/boot/firmware/cmdline.txt``.

In [ ]:
sudo nano /boot/firmware/cmdline.txt

**Step 3** ::: On the same line, at the very end, add these parameters **(without pressing Enter)**:

In [ ]:
quiet splash plymouth.ignore-serial-consoles logo.nologo vt.global_cursor_default=0

VVV It should look something like this VVV

root=PARTUUID=SOMETHING rootfstype=ext4 fsck.repair=yes rootwait quiet splash plymouth.ignore-serial-consoles logo.nologo vt.global_cursor_default=0

**Step 4** ::: Configure the Plymouth theme and unmask the service.

**Step 5 (CRITICAL STEP)** ::: After all of these commands of the splash screen, regenerate the initramfs.
###### Note ::: This has to be done so that Plymouth starts early enough to show the picture. Without this command, the kernel would fill the screen back with the Raspberry Pi OS picture.

In [ ]:
sudo update-initramfs -u

**Step 6** ::: Reboot.

In [ ]:
sudo reboot

__**Touch Screen Adjustment for Wayland**__

**Step 1** ::: Open `/etc/udev/rules.d/99-touchscreen-cal.rules`.

In [ ]:
sudo nano /etc/udev/rules.d/99-touchscreen-cal.rules

**Step 2** ::: Paste this line inside that file.

In [ ]:
SUBSYSTEM=="input", KERNEL=="event*", ENV{LIBINPUT_CALIBRATION_MATRIX}="-1 0 1 0 -1 1"

**Step 3** ::: Then reload the rules and trigger them.

In [ ]:
sudo udevadm control --reload-rules && sudo udevadm trigger

**Step 4** ::: Reboot.

In [ ]:
sudo reboot

##### __**Troubleshooting for Splash Screen Configuration**__

**Problem 1** ::: Splash only appears on shutdown, not on boot.

Make sure you have made the Step 4 of Raspberry Pi Configurations properly.


**Problem 2** ::: The ``cmdline.txt`` got corrupted and the system won't boot.

If you accidentally edit the PARTUUID or break the cmdline.txt line, the system may fail to boot. 
To fix it without access to the system:

- **Step 1** ::: Remove the SD card and connect it to your PC (it will appear as the bootfs partition on Windows — FAT32, directly accessible).

- **Step 2** ::: Open the `cmdline.txt` file in the root of that partition.

- **Step 3** ::: Restore the content. If you don't know the original PARTUUID, replace it with /dev/mmcblk0p2, as follows:

In [ ]:
console=serial0,115200 console=tty1 root=/dev/mmcblk0p2 rootfstype=ext4 fsck.repair=yes rootwait quiet splash plymouth.ignore-serial-consoles logo.nologo vt.global_cursor_default=0

- **Step 4** ::: Save, eject the SD card and test.

**Problem 3** ::: The text messages still appear during boot.

- **Step 1** ::: Confirm that ``cmdline.txt`` actually contains ``quiet`` and ``splash``:

In [ ]:
cat /boot/firmware/cmdline.txt

- **Step 2** ::: If it doesn't have, please add it **or** replace the whole line with the line exposed on the Step 3 of the Problem 2 (recommended).

- **Step 3** ::: Save and test.

**Problem 4** ::: Plymouth doesn't appear at all.

- **Step 1** ::: Check if the service is masked by doing the following command:

In [ ]:
systemctl status plymouth-start

- **Step 2** ::: If Plymouth shows masked, unmask it:

In [ ]:
sudo systemctl unmask plymouth-start.service
sudo update-initramfs -u

- **Step 3** ::: Reboot the system.

In [ ]:
sudo reboot